# Electronic Waste Management - EDA

> Exploratory Data Analysis focused on understanding data quality, structure, and behavior.

This notebook is focused on:
- understanding the structure of the data,
- detecting patterns and relationships,
- identifying outliers and anomalies,
- checking missing values,
- understanding distributions of variables.

In [ ]:
# Core imports
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

plt.style.use('ggplot')
RANDOM_STATE = 42

In [ ]:
# Load dataset
df = pd.read_csv('electronics_pricing_dataset.csv')

print(f'Shape: {df.shape}')
print(f'Rows: {df.shape[0]:,} | Columns: {df.shape[1]}')
print(df.columns)
df.head()

In [ ]:
# Schema and quality checks
print('--- Column types ---')
display(df.dtypes.to_frame('dtype'))

missing = df.isna().sum().sort_values(ascending=False)
duplicates = df.duplicated().sum()

print('--- Data quality summary ---')
print(f'Total missing values: {int(missing.sum())}')
print(f'Duplicate rows: {int(duplicates)}')

if missing.sum() > 0:
    display(missing[missing > 0].to_frame('missing_count'))
else:
    print('No missing values found.')

In [ ]:
# Separate numeric/categorical columns
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include=['object', 'string', 'category']).columns.tolist()

print('Numeric columns:', num_cols)
print('Categorical columns:', cat_cols)

display(df[num_cols].describe().T)

# Useful quantiles for heavy-tailed price behavior
q = df[['Original_Price', 'Current_Price']].quantile([0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99])
display(q)

In [ ]:
# Categorical profile
for c in cat_cols:
    print(f'\n=== {c} ===')
    print(f'Unique values: {df[c].nunique()}')
    display(df[c].value_counts(dropna=False).head(15).to_frame('count'))

In [ ]:
# Univariate numeric distributions
plot_cols = [
    'Original_Price', 'Current_Price', 'Used_Duration', 'Expiry_Years',
    'Condition', 'Build_Quality'
 ]

fig, axes = plt.subplots(3, 2, figsize=(14, 12))
axes = axes.flatten()

for i, c in enumerate(plot_cols):
    axes[i].hist(df[c], bins=35, color='steelblue', alpha=0.85, edgecolor='black')
    axes[i].set_title(f'{c} Distribution')
    axes[i].set_xlabel(c)
    axes[i].set_ylabel('Count')

# Hide any extra subplot axes
for j in range(len(plot_cols), len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix (numeric only)
corr_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
corr = df[corr_cols].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(12, 8))
im = ax.imshow(corr.values, cmap='coolwarm', aspect='auto', vmin=-1, vmax=1)

# Major ticks and labels
ax.set_xticks(np.arange(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=90)
ax.set_yticks(np.arange(len(corr.columns)))
ax.set_yticklabels(corr.columns)
ax.set_title('Correlation Heatmap (Numeric Features)')

# Draw cell gridlines so each color sits in a distinct box
ax.set_xticks(np.arange(-0.5, len(corr.columns), 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(corr.columns), 1), minor=True)
ax.grid(which='minor', color='white', linestyle='-', linewidth=1.2)
ax.tick_params(which='minor', bottom=False, left=False)

cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Correlation')
plt.tight_layout()
plt.show()

display(corr['Current_Price'].sort_values(ascending=False).to_frame('corr_with_Current_Price'))

In [ ]:
# Product-level pricing behavior
prod_stats = (
    df.groupby('Product_Type', as_index=False)
      .agg(
           count=('Product_Type', 'size'),
           avg_original_price=('Original_Price', 'mean'),
           avg_current_price=('Current_Price', 'mean'),
           median_current_price=('Current_Price', 'median'),
           avg_used_duration=('Used_Duration', 'mean'),
           avg_condition=('Condition', 'mean')
       )
      .sort_values('median_current_price', ascending=False)
)

display(prod_stats)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

tmp = prod_stats.sort_values('median_current_price', ascending=True)
axes[0].barh(tmp['Product_Type'], tmp['median_current_price'], color='teal')
axes[0].set_title('Median Current Price by Product Type')
axes[0].set_xlabel('Median Current Price')

tmp2 = prod_stats.sort_values('avg_used_duration', ascending=True)
axes[1].barh(tmp2['Product_Type'], tmp2['avg_used_duration'], color='tomato')
axes[1].set_title('Average Used Duration by Product Type')
axes[1].set_xlabel('Average Used Duration (years)')

plt.tight_layout()
plt.show()

In [ ]:
# Usage pattern and condition analysis
usage_stats = (
    df.groupby('Usage_Pattern', as_index=False)
      .agg(
           count=('Usage_Pattern', 'size'),
           avg_used_duration=('Used_Duration', 'mean'),
           avg_current_price=('Current_Price', 'mean'),
           median_current_price=('Current_Price', 'median'),
           avg_condition=('Condition', 'mean')
       )
      .sort_values('avg_current_price', ascending=False)
)
display(usage_stats)

cond_stats = (
    df.groupby('Condition', as_index=False)
      .agg(
           count=('Condition', 'size'),
           avg_current_price=('Current_Price', 'mean'),
           median_current_price=('Current_Price', 'median'),
           q1_current_price=('Current_Price', lambda s: s.quantile(0.25)),
           q3_current_price=('Current_Price', lambda s: s.quantile(0.75))
       )
      .sort_values('Condition')
)
display(cond_stats)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(cond_stats['Condition'], cond_stats['median_current_price'], marker='o', linewidth=2, color='purple')
ax.set_title('Condition vs Median Current Price')
ax.set_xlabel('Condition')
ax.set_ylabel('Median Current Price')
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Outlier and anomaly checks (IQR method)
check_cols = ['Original_Price', 'Current_Price', 'Used_Duration', 'Expiry_Years']
outlier_rows = pd.Series(False, index=df.index)
outlier_summary = []

for col in check_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    mask = (df[col] < lower) | (df[col] > upper)
    outlier_rows |= mask
    outlier_summary.append({
        'feature': col,
        'lower_bound': lower,
        'upper_bound': upper,
        'outlier_count': int(mask.sum()),
        'outlier_pct': round(mask.mean() * 100, 2)
    })

outlier_df = pd.DataFrame(outlier_summary).sort_values('outlier_pct', ascending=False)
display(outlier_df)

print(f'Total rows flagged as outlier in at least one checked feature: {int(outlier_rows.sum())} ({outlier_rows.mean()*100:.2f}%)')
print(f'Potential anomaly rows where Used_Duration > Expiry_Years: {int((df["Used_Duration"] > df["Expiry_Years"]).sum())}')

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df['Used_Duration'], df['Current_Price'], s=10, alpha=0.25, color='steelblue')
ax.set_title('Current Price vs Used Duration (Anomaly View)')
ax.set_xlabel('Used Duration (years)')
ax.set_ylabel('Current Price')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()